# 01 - Data collection and understanding


## Step 0 - Problem Definition

### Chosen target variable

We selected the Clear Sky Index as the modeling variable due to its versatility in addressing various issues associated with using raw power output directly—such as those related to celestial mechanics and atmospheric physics.

The main advantages of modeling with the Clear Sky Index include:
- Elimination of massive, non-stationary diurnal and annual seasonality.
- Greater efficiency regarding parameters and the learning approach.
- Reduced error variance (as values ​​fluctuate within a bounded range).
- Enhanced generalization and knowledge transfer (e.g., the ability to reuse the model if capacity is expanded).

### Chosen prediction horizon

We set the prediction horizon to 48 hours during the project's prototyping phase to validate the data and the variable's behavior. Subsequently, this will be extended to 7 days, taking into account the drastic change in the nature of the features and addressing the problem through NWP-based predictions.

### Chosen temporal granularity

We defined a granularity of one hour, suitable for medium-term horizons (1 to 7 days). This aligns with the Spanish wholesale market and its hourly scheduling; it also smooths out fluctuations that are impossible to predict seven days in advance while preserving accuracy regarding solar ramp events.

## Step 1 - Raw data loading

In [287]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [288]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from src.config import COPERNICUS_DATA_FILENAME, PVGIS_DATA_FILENAME
from src.data.ingest import load_raw_cloudiness_data, load_raw_irradiance_data
from src.data.validation import (
    get_duplicate_rows,
    has_missing_rows,
    has_timestamp_gaps,
)

irradiance_data = load_raw_irradiance_data(PVGIS_DATA_FILENAME)
cloudiness_data = load_raw_cloudiness_data(COPERNICUS_DATA_FILENAME)

## Step 2 - First look

### Cloudiness dataset

In [290]:
cloudiness_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 180648 entries, 0 to 180647
Data columns (total 20 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   valid_time  180648 non-null  datetime64[us]
 1   v100        180648 non-null  float64       
 2   v10         180648 non-null  float64       
 3   d2m         180648 non-null  float64       
 4   t2m         180648 non-null  float64       
 5   blh         180648 non-null  float64       
 6   cbh         136164 non-null  float64       
 7   msl         180648 non-null  float64       
 8   sst         0 non-null       float64       
 9   skt         180648 non-null  float64       
 10  slhf        180648 non-null  float64       
 11  sp          180648 non-null  float64       
 12  sshf        180648 non-null  float64       
 13  ssrd        180648 non-null  float64       
 14  strd        180648 non-null  float64       
 15  tcc         180648 non-null  float64       
 16  tp          1

In [291]:
is_cloudiness_missing_rows = has_missing_rows(cloudiness_data, 'valid_time')

display("There are missing rows in the cloudiness data." if is_cloudiness_missing_rows else "There are no missing rows in the cloudiness data.")


'There are no missing rows in the cloudiness data.'

The Copernicus dataset is not missing any row.

In [292]:
duplicated_rows = get_duplicate_rows(cloudiness_data, 'valid_time')
display(f"There are duplicated rows in the cloudiness data. Number of duplicates: {len(duplicated_rows)}" if not duplicated_rows.empty else "There are no duplicated rows in the cloudiness data.")

'There are no duplicated rows in the cloudiness data.'

There are no duplicated timestamps.

In [293]:
exists_hourly_diff = has_timestamp_gaps(cloudiness_data, 'valid_time')

display("There's a gap in the cloudiness data." if exists_hourly_diff else "There are no gaps in the cloudiness data.")

'There are no gaps in the cloudiness data.'

The difference between a timestamp and the previous one is exactly 1 hour, which means there's no gap between the data rows.

In [294]:
non_null_cbh_percentage = (cloudiness_data['cbh'].count() / len(cloudiness_data)) * 100
total_null_cbh_percentage = 100 - non_null_cbh_percentage

print('Percentage of missing cells in Cloud Base Height column data:', total_null_cbh_percentage.round(2), '%')

Percentage of missing cells in Cloud Base Height column data: 24.62 %


`sst` (Sea Surface Temperature) does not contain any value. It is not critical information to calculate the Clear Sky Index, so it's fit for further removal.

`cbh` (Cloud Base Height) is missing ~24.62% of its data. Although cloud height can affect density—and, consequently, the amount of sky obscured—it is not a significant variable and can be disregarded later.

In [295]:
cloudiness_data.describe()

,valid_time,v100,v10,d2m,t2m,blh,cbh,msl,sst,skt,slhf,sp,sshf,ssrd,strd,tcc,tp,fdir,latitude,longitude
count,180648,180648.000000,180648.000000,180648.000000,180648.000000,180648.000000,136164.000000,180648.000000,0.0,180648.000000,1.806480e+05,180648.000000,1.806480e+05,1.806480e+05,1.806480e+05,180648.000000,180648.000000,1.806480e+05,180648.0,180648.00
mean,2016-04-21 11:30:00,0.264484,0.116441,284.761796,289.242988,363.622462,2127.244135,101664.775953,NaN,289.673189,-2.092601e+05,99607.481019,-1.058324e+05,6.589256e+05,1.183160e+06,0.485599,0.000071,4.596590e+05,41.5,2.25
min,2006-01-01 00:00:00,-14.761261,-10.095184,260.081800,270.632260,10.968915,29.346115,98125.060000,NaN,272.733550,-1.322378e+06,96117.050000,-1.158995e+06,-1.901566e+00,6.501370e+05,0.000000,0.000000,0.000000e+00,41.5,2.25
25%,2011-02-25 17:45:00,-1.398476,-1.297619,280.170282,284.165680,100.755304,431.801663,101297.940000,NaN,283.537990,-3.158758e+05,99254.834500,-1.524578e+05,0.000000e+00,1.060967e+06,0.092346,0.000000,0.000000e+00,41.5,2.25
50%,2016-04-21 11:30:00,0.145897,-0.142586,284.761080,288.964750,288.433915,1013.413900,101682.810000,NaN,289.197720,-1.350105e+05,99635.867500,-1.698550e+04,3.296000e+04,1.189939e+06,0.465012,0.000000,3.584000e+03,41.5,2.25
75%,2021-06-16 05:15:00,2.026775,1.479176,289.919062,294.356750,546.506697,2829.673550,102063.690000,NaN,294.918910,-5.676200e+04,99999.470000,3.886250e+03,1.218880e+06,1.305808e+06,0.885803,0.000008,7.479200e+05,41.5,2.25
max,2026-08-10 23:00:00,14.282288,9.590134,298.401670,307.998540,3442.432400,14455.278000,104049.810000,NaN,314.045780,4.511500e+04,101906.250000,4.707780e+05,3.664064e+06,1.592084e+06,1.000000,0.012948,3.225664e+06,41.5,2.25
std,NaN,2.690851,2.030372,6.037097,6.422014,317.105845,2502.310561,693.808834,NaN,7.380560,1.980914e+05,680.066960,1.919113e+05,9.171165e+05,1.534523e+05,0.376779,0.000320,7.173230e+05,0.0,0.00


`ssrd` (surface downward solar radiation) has a minimum value of -1.9. This value is physically impossible, so it is likely a numerical artifact of the reanalysis model. It should be set to the minimum valid value (0) during the cleaning process (02_eda).

The `latitude` and `longitude` values ​​do not change, so they are candidates for removal later on.

Let's focus on `tcc` (Total cloud cover):

In [296]:
cloudiness_data['tcc'].describe()

count    180648.000000
mean          0.485599
std           0.376779
min           0.000000
25%           0.092346
50%           0.465012
75%           0.885803
max           1.000000
Name: tcc, dtype: float64

The `tcc` (Total Cloud Cover) range is between 0 and 1. The cloudiness tends to distribute toward both ends of the range: the lowest 25% of values fall below 0.092, while the top 25% exceed 0.886. Might be interesting to explore the hypothesis of more cloudy periods of time later (in the 02_eda notebook). This is a feature that our model can greatly benefit from, helping it differentiate between cloudy and clear sky hours.

In [297]:

display(cloudiness_data['valid_time'].min())
display(cloudiness_data['valid_time'].max())

Timestamp('2006-01-01 00:00:00')

Timestamp('2026-08-10 23:00:00')

The dataset Copernicus Climate Data Store includes data from 2006-01-01 to 2026-08-10, which is a total of over 20 years of data. We will shorten the dataset to only work with the data contained between the years 2006-2023.
Later, we will use this data to train and test our machine learning models for solar irradiance forecasting.

### Irradiance dataset

In [298]:
irradiance_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 166536 entries, 0 to 166535
Data columns (total 9 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   time    166536 non-null  datetime64[us]
 1   P       166536 non-null  float64       
 2   Gb(i)   166536 non-null  float64       
 3   Gd(i)   166536 non-null  float64       
 4   Gr(i)   166536 non-null  float64       
 5   H_sun   166536 non-null  float64       
 6   T2m     166536 non-null  float64       
 7   WS10m   166536 non-null  float64       
 8   Int     166536 non-null  float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 11.4 MB


The PVGIS dataset does not contain any missing cells.

In [299]:
is_irradiance_missing_rows = has_missing_rows(irradiance_data, 'time')

display("There are missing rows in the irradiance data." if is_irradiance_missing_rows else "There are no missing rows in the irradiance data.")


'There are no missing rows in the irradiance data.'

The PVGIS dataset is not missing any row.

In [300]:
duplicated_rows = get_duplicate_rows(irradiance_data, 'time')
display(f"There are duplicated rows in the irradiance data. Number of duplicates: {len(duplicated_rows)}" if not duplicated_rows.empty else "There are no duplicated rows in the irradiance data.")

'There are no duplicated rows in the irradiance data.'

There are no duplicated timestamps.

In [301]:
exists_hourly_diff = has_timestamp_gaps(irradiance_data, 'time')

display("There's a gap in the irradiance data." if exists_hourly_diff else "There are no gaps in the irradiance data.")

'There are no gaps in the irradiance data.'

The difference between a timestamp and the previous one is exactly 1 hour, which means there's no gap between the data rows.

In [302]:
irradiance_data.describe()

,time,P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,Int
count,166536,166536.000000,166536.000000,166536.000000,166536.000000,166536.000000,166536.000000,166536.000000,166536.0
mean,2014-07-02 12:00:00,180.002006,164.422634,58.138489,4.087388,15.132049,17.288311,4.010770,0.0
min,2005-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,-5.630000,2.490000,0.000000,0.0
25%,2009-10-01 18:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,12.910000,2.140000,0.0
50%,2014-07-02 12:00:00,0.000000,0.000000,5.260000,0.140000,0.200000,16.580000,3.380000,0.0
75%,2019-04-02 05:45:00,369.285000,292.972500,119.160000,7.660000,27.972500,22.030000,5.100000,0.0
max,2023-12-31 23:30:00,911.980000,975.580000,392.710000,21.740000,71.440000,30.180000,37.790000,0.0
std,NaN,251.298950,246.022921,73.002041,5.696382,20.232038,5.214038,3.148401,0.0


The `mean` for `P` (Photovoltaic power output) is ~180W.

The `mean` for `Gb(i)` is ~164.42W/m².

The `median` for both `P` and `Gb(i)` is 0, which means there's more than half the rows with 0. This makes sense because of the natural shape of the hourly data, where the solar irradiance tends to 0 at sunrise and sunset, and reaches 0 during night-time.

All values ​​in the `Int` column are set to 0, which means that none of the values ​​have been reconstructed. This is a very good indicator of high-quality data.

In [303]:
irradiance_data[(irradiance_data['H_sun'] == irradiance_data['H_sun'].min())]

,time,P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,Int
133373,2020-03-20 05:30:00,0.0,0.0,0.0,0.0,-5.63,11.06,3.93,0.0


The `H_sun` `min` value is -5.63º at 2020-03-20 05:30:00 (06:30 CET). This value is the average of an hour that includes the sunrise transition. This indicates that `H_sun` does not indicate how deep into the night-time it is, but it can help identify the transition hours (sunrise/sunset).

In [304]:

display(irradiance_data['time'].min())
display(irradiance_data['time'].max())

Timestamp('2005-01-01 00:30:00')

Timestamp('2023-12-31 23:30:00')

The PVGIS dataset contains older data than the Copernicus Climate Data Store, but its most recent data point is also much older than that of the latter. Therefore, we will truncate the datasets to work with a time interval common to both.